In [34]:
import pandas as pd
import json
import re
import time
import os
from pathlib import Path
from collections import defaultdict

In [35]:
NOTES_PATH = Path('../dane/unique_specific_notes.csv')
ACCORDS_PATH = Path('../dane/unique_accords.csv')
PERFUMES_PATH = Path('../dane/fra_cleaned_final.csv')

notes_df = pd.read_csv(NOTES_PATH)
accords_df = pd.read_csv(ACCORDS_PATH)
perfumes_df = pd.read_csv(PERFUMES_PATH, sep=';', encoding='utf-8-sig')

In [36]:
notes = (
    notes_df['Note Name']
    .dropna()
    .astype(str)
    .str.strip()
)

notes = sorted(set(note for note in notes if note))

notes_normalized = {
    note: " ".join(note.casefold().split())
    for note in notes
}

In [37]:
from itertools import combinations

note_set = set(notes_normalized.values())

subset_candidate_rows = []

for original_note in notes:
    normalized_note = notes_normalized[original_note]
    tokens = normalized_note.split()

    if len(tokens) < 2:
        continue

    for size in range(1, len(tokens)):
        for indices in combinations(range(len(tokens)), size):
            candidate = " ".join(tokens[i] for i in indices)

            if candidate in note_set and candidate != normalized_note:
                subset_candidate_rows.append({
                    "specific_note": original_note,
                    "possible_family": candidate,
                    "specific_word_count": len(tokens),
                    "family_word_count": size,
                    "removed_words": " ".join(
                        token for i, token in enumerate(tokens)
                        if i not in indices
                    )
                })

subset_candidates_df = pd.DataFrame(subset_candidate_rows)

if not subset_candidates_df.empty:
    subset_candidates_df = (
        subset_candidates_df
        .drop_duplicates()
        .sort_values(
            ["specific_note", "family_word_count"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
    )

print("Liczba nazw źródłowych:", len(notes))
print("Liczba par nazwa–kandydat:", len(subset_candidates_df))
print(
    "Liczba nazw mających przynajmniej jednego kandydata:",
    subset_candidates_df["specific_note"].nunique()
    if not subset_candidates_df.empty else 0
)

display(
    subset_candidates_df[
        subset_candidates_df["specific_note"].isin([
            "chinese black tea",
            "madagascar vanilla",
            "calabrian bergamot",
            "white musk",
            "water fruit"
        ])
    ]
)

Liczba nazw źródłowych: 1664
Liczba par nazwa–kandydat: 859
Liczba nazw mających przynajmniej jednego kandydata: 724


,specific_note,possible_family,specific_word_count,family_word_count,removed_words
146,calabrian bergamot,bergamot,2,1,calabrian
183,chinese black tea,black,3,1,chinese tea
184,chinese black tea,tea,3,1,chinese black
185,chinese black tea,chinese tea,3,2,black
186,chinese black tea,black tea,3,2,chinese
460,madagascar vanilla,vanilla,2,1,madagascar
809,white musk,musk,2,1,white


In [38]:
import numpy as np

if subset_candidates_df.empty:
    print("Nie znaleziono kandydatów do normalizacji.")
else:
    subset_candidate_summary = (
        subset_candidates_df
        .groupby("specific_note")
        .agg(
            number_of_candidates=("possible_family", "nunique"),
            candidates=("possible_family", lambda x: " | ".join(
                sorted(set(x), key=lambda name: (len(name.split()), name))
            ))
        )
        .reset_index()
    )

    subset_candidate_summary["review_status"] = np.where(
        subset_candidate_summary["number_of_candidates"] == 1,
        "Jeden kandydat — do oceny",
        "Wiele kandydatów — wymaga porównania"
    )

    subset_candidate_summary = subset_candidate_summary.sort_values(
        ["number_of_candidates", "specific_note"],
        ascending=[False, True]
    ).reset_index(drop=True)

    print("Liczba nazw z kandydatami:", len(subset_candidate_summary))
    print("\nRozkład liczby kandydatów:")
    display(
        subset_candidate_summary["number_of_candidates"]
        .value_counts()
        .sort_index()
        .rename_axis("number_of_candidates")
        .reset_index(name="number_of_notes")
    )

    print("\nPrzykłady wymagające porównania:")
    display(subset_candidate_summary.head(40))

Liczba nazw z kandydatami: 724

Rozkład liczby kandydatów:


,number_of_candidates,number_of_notes
0,1,619
1,2,82
2,3,16
3,4,7



Przykłady wymagające porównania:


,specific_note,number_of_candidates,candidates,review_status
0,chinese black tea,4,black | tea | black tea | chinese tea,Wiele kandydatów — wymaga porównania
1,darjeeling black tea,4,black | tea | black tea | darjeeling tea,Wiele kandydatów — wymaga porównania
2,silk vine or milk broom,4,broom | milk | silk | vine,Wiele kandydatów — wymaga porównania
3,white chocolate truffle,4,chocolate | truffle | chocolate truffle | whit...,Wiele kandydatów — wymaga porównania
4,white ginger lily,4,ginger | lily | white ginger | white lily,Wiele kandydatów — wymaga porównania
5,white mimosa honey,4,honey | mimosa | white honey | white mimosa,Wiele kandydatów — wymaga porównania
6,white peach blossom,4,peach | peach blossom | white blossom | white ...,Wiele kandydatów — wymaga porównania
7,apple tree blossom,3,apple | apple blossom | apple tree,Wiele kandydatów — wymaga porównania
8,bitter orange blossom,3,orange | bitter orange | orange blossom,Wiele kandydatów — wymaga porównania
9,black locust honey,3,black | honey | black locust,Wiele kandydatów — wymaga porównania


In [39]:
import spacy

In [40]:
nlp = spacy.load("en_core_web_sm")

print("spaCy:", spacy.__version__)
print("Model:", nlp.meta["name"])
print("Język:", nlp.meta["lang"])

spaCy: 3.8.14
Model: core_web_sm
Język: en


In [41]:
import pandas as pd

test_notes = [
    "calabrian bergamot",
    "madagascar vanilla",
    "chinese black tea",
    "orange blossom",
    "water fruit",
    "white musk",
    "indian jasmine",
    "virginia cedar",
    "black pepper",
    "agarwood",
    "oud"
]

rows = []

for note in test_notes:
    doc = nlp(note)

    for token in doc:
        rows.append({
            "full_note": note,
            "token": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "dependency": token.dep_,
            "head": token.head.text
        })

spacy_test_df = pd.DataFrame(rows)

display(spacy_test_df)

,full_note,token,lemma,pos,dependency,head
0,calabrian bergamot,calabrian,calabrian,PROPN,amod,bergamot
1,calabrian bergamot,bergamot,bergamot,NOUN,ROOT,bergamot
2,madagascar vanilla,madagascar,madagascar,ADJ,amod,vanilla
3,madagascar vanilla,vanilla,vanilla,NOUN,ROOT,vanilla
4,chinese black tea,chinese,chinese,ADJ,amod,tea
5,chinese black tea,black,black,ADJ,amod,tea
6,chinese black tea,tea,tea,NOUN,ROOT,tea
7,orange blossom,orange,orange,PROPN,compound,blossom
8,orange blossom,blossom,blossom,PROPN,ROOT,blossom
9,water fruit,water,water,NOUN,compound,fruit


In [42]:
all_docs = list(nlp.pipe(notes, batch_size=128))

analysis_rows = []

for note, doc in zip(notes, all_docs):
    analysis_rows.append({
        "note": note,
        "tokens": len(doc),
        "adjectives": " | ".join(
            token.text for token in doc if token.pos_ == "ADJ"
        ),
        "nouns": " | ".join(
            token.text for token in doc if token.pos_ == "NOUN"
        ),
        "proper_nouns": " | ".join(
            token.text for token in doc if token.pos_ == "PROPN"
        ),
        "lemmas": " ".join(token.lemma_ for token in doc),
        "pos_sequence": " ".join(token.pos_ for token in doc)
    })

spacy_notes_df = pd.DataFrame(analysis_rows)

print("Przeanalizowano nut:", len(spacy_notes_df))
print("\nNajczęstsze struktury gramatyczne:")
display(
    spacy_notes_df["pos_sequence"]
    .value_counts()
    .head(20)
    .rename_axis("pos_sequence")
    .reset_index(name="count")
)

print("\nPrzykłady analizy:")
display(spacy_notes_df.head(30))

Przeanalizowano nut: 1664

Najczęstsze struktury gramatyczne:


,pos_sequence,count
0,NOUN,339
1,PROPN,241
2,PROPN NOUN,200
3,ADJ NOUN,196
4,PROPN PROPN,164
5,NOUN NOUN,162
6,VERB,70
7,VERB NOUN,32
8,PROPN VERB,29
9,ADJ,24



Przykłady analizy:


,note,tokens,adjectives,nouns,proper_nouns,lemmas,pos_sequence
0,absinthe,1,,,,absinthe,ADV
1,acai berry,2,,acai | berry,,acai berry,NOUN NOUN
2,accord eudora,2,,,accord | eudora,accord eudora,PROPN PROPN
3,acerola,1,,,acerola,acerola,PROPN
4,acerola blossom,2,,blossom,acerola,acerola blossom,PROPN NOUN
5,acetylfuran,1,acetylfuran,,,acetylfuran,ADJ
6,acácia,1,,acácia,,acácia,NOUN
7,african freesia petals,3,,freesia | petals,african,african freesia petal,PROPN NOUN NOUN
8,african geranium,2,african,geranium,,african geranium,ADJ NOUN
9,african ginger,2,african,ginger,,african ginger,ADJ NOUN


In [43]:
import pandas as pd

spacy_candidate_rows = []

for note, doc in zip(notes, all_docs):
    root = next(
        (token for token in doc if token.dep_ == "ROOT"),
        None
    )

    candidate = root.lemma_.casefold().strip() if root else note.casefold()

    spacy_candidate_rows.append({
        "note": note,
        "root": root.text if root else None,
        "root_lemma": candidate,
        "pos_sequence": " ".join(token.pos_ for token in doc),
        "tokens": len(doc),
    })

spacy_candidates_df = pd.DataFrame(spacy_candidate_rows)

available_notes = set(notes_normalized.values())

spacy_candidates_df["candidate_exists"] = (
    spacy_candidates_df["root_lemma"].isin(available_notes)
)

display(spacy_candidates_df.head(40))

,note,root,root_lemma,pos_sequence,tokens,candidate_exists
0,absinthe,absinthe,absinthe,ADV,1,True
1,acai berry,berry,berry,NOUN NOUN,2,False
2,accord eudora,eudora,eudora,PROPN PROPN,2,False
3,acerola,acerola,acerola,PROPN,1,True
4,acerola blossom,blossom,blossom,PROPN NOUN,2,False
5,acetylfuran,acetylfuran,acetylfuran,ADJ,1,True
6,acácia,acácia,acácia,NOUN,1,True
7,african freesia petals,petals,petal,PROPN NOUN NOUN,3,False
8,african geranium,geranium,geranium,ADJ NOUN,2,True
9,african ginger,ginger,ginger,ADJ NOUN,2,True


In [44]:
multiword_candidates = spacy_candidates_df[
    (spacy_candidates_df["tokens"] > 1)
    & (spacy_candidates_df["candidate_exists"])
    & (spacy_candidates_df["note"].str.casefold()
       != spacy_candidates_df["root_lemma"])
].copy()

display(
    multiword_candidates[
        ["note", "root", "root_lemma", "pos_sequence"]
    ].head(100)
)

print("Liczba wszystkich nut:", len(spacy_candidates_df))
print(
    "Kandydaci wielowyrazowi z istniejącą krótszą nazwą:",
    len(multiword_candidates)
)

,note,root,root_lemma,pos_sequence
8,african geranium,geranium,geranium,ADJ NOUN
9,african ginger,ginger,ginger,ADJ NOUN
11,african violet,violet,violet,PROPN NOUN
19,algerian geranium,geranium,geranium,ADJ NOUN
24,almond milk,milk,milk,PROPN NOUN
...,...,...,...,...
387,clary sage,sage,sage,ADJ NOUN
407,coconut milk,milk,milk,NOUN NOUN
412,coffee co2,coffee,coffee,NOUN NOUN
416,columbian coffee,coffee,coffee,ADJ NOUN


Liczba wszystkich nut: 1664
Kandydaci wielowyrazowi z istniejącą krótszą nazwą: 405


In [45]:
import pandas as pd

subset_candidates = subset_candidates_df[
    ["specific_note", "possible_family"]
].copy()

subset_candidates["source"] = "word_combinations"

spacy_candidates = spacy_candidates_df[
    spacy_candidates_df["candidate_exists"]
    & (spacy_candidates_df["tokens"] > 1)
    & (
        spacy_candidates_df["note"].str.casefold()
        != spacy_candidates_df["root_lemma"]
    )
][["note", "root_lemma"]].rename(columns={
    "note": "specific_note",
    "root_lemma": "possible_family"
}).copy()

spacy_candidates["source"] = "spacy_root"

combined_candidates = pd.concat(
    [subset_candidates, spacy_candidates],
    ignore_index=True
)

combined_candidates = (
    combined_candidates
    .groupby(
        ["specific_note", "possible_family"],
        as_index=False
    )
    .agg(
        sources=("source", lambda x: " | ".join(sorted(set(x))))
    )
    .sort_values(
        ["specific_note", "possible_family"]
    )
    .reset_index(drop=True)
)

print("Pary z metody kombinacji:", len(subset_candidates))
print("Pary z metody spaCy:", len(spacy_candidates))
print("Unikalne pary po połączeniu:", len(combined_candidates))

display(combined_candidates.head(40))

Pary z metody kombinacji: 859
Pary z metody spaCy: 405
Unikalne pary po połączeniu: 867


,specific_note,possible_family,sources
0,acerola blossom,acerola,word_combinations
1,african freesia petals,freesia,word_combinations
2,african freesia petals,freesia petals,word_combinations
3,african geranium,geranium,spacy_root | word_combinations
4,african ginger,ginger,spacy_root | word_combinations
5,african orange flower,orange,word_combinations
6,african violet,violet,spacy_root | word_combinations
7,agave nectar,agave,word_combinations
8,algerian geranium,geranium,spacy_root | word_combinations
9,almond blossom,almond,word_combinations


In [46]:
test_notes = [
    "chinese black tea",
    "madagascar vanilla",
    "calabrian bergamot",
    "white musk",
    "water fruit",
    "orange blossom",
    "virginia cedar",
    "agarwood",
    "oud"
]

display(
    combined_candidates[
        combined_candidates["specific_note"].str.casefold().isin(
            test_notes
        )
    ].sort_values(
        ["specific_note", "possible_family"]
    )
)

,specific_note,possible_family,sources
146,calabrian bergamot,bergamot,spacy_root | word_combinations
183,chinese black tea,black,word_combinations
184,chinese black tea,black tea,word_combinations
185,chinese black tea,chinese tea,word_combinations
186,chinese black tea,tea,spacy_root | word_combinations
465,madagascar vanilla,vanilla,spacy_root | word_combinations
529,orange blossom,orange,word_combinations
766,virginia cedar,cedar,spacy_root | word_combinations
817,white musk,musk,spacy_root | word_combinations


In [47]:
# Dodaj informacje ze spaCy do wspólnej tabeli

root_lookup = (
    spacy_candidates_df
    .set_index("note")["root_lemma"]
    .to_dict()
)

combined_candidates["spacy_root"] = (
    combined_candidates["specific_note"]
    .map(root_lookup)
)

combined_candidates["matches_spacy_root"] = (
    combined_candidates["possible_family"]
    == combined_candidates["spacy_root"]
)

combined_candidates["specific_word_count"] = (
    combined_candidates["specific_note"].str.split().str.len()
)

combined_candidates["family_word_count"] = (
    combined_candidates["possible_family"].str.split().str.len()
)

combined_candidates["is_shorter"] = (
    combined_candidates["family_word_count"]
    < combined_candidates["specific_word_count"]
)

display(
    combined_candidates[
        combined_candidates["specific_note"].str.casefold().isin([
            "calabrian bergamot",
            "chinese black tea",
            "madagascar vanilla",
            "orange blossom",
            "virginia cedar",
            "white musk"
        ])
    ].sort_values(["specific_note", "possible_family"])
)

,specific_note,possible_family,sources,spacy_root,matches_spacy_root,specific_word_count,family_word_count,is_shorter
146,calabrian bergamot,bergamot,spacy_root | word_combinations,bergamot,True,2,1,True
183,chinese black tea,black,word_combinations,tea,False,3,1,True
184,chinese black tea,black tea,word_combinations,tea,False,3,2,True
185,chinese black tea,chinese tea,word_combinations,tea,False,3,2,True
186,chinese black tea,tea,spacy_root | word_combinations,tea,True,3,1,True
465,madagascar vanilla,vanilla,spacy_root | word_combinations,vanilla,True,2,1,True
529,orange blossom,orange,word_combinations,blossom,False,2,1,True
766,virginia cedar,cedar,spacy_root | word_combinations,cedar,True,2,1,True
817,white musk,musk,spacy_root | word_combinations,musk,True,2,1,True


In [48]:
combined_candidates["review_status"] = "Wymaga weryfikacji"

combined_candidates.loc[
    combined_candidates["matches_spacy_root"]
    & combined_candidates["is_shorter"]
    & combined_candidates["sources"].str.contains(
        "word_combinations", regex=False
    ),
    "review_status"
] = "Priorytetowy kandydat"

display(
    combined_candidates["review_status"]
    .value_counts()
    .rename_axis("review_status")
    .reset_index(name="number_of_candidates")
)

display(
    combined_candidates[
        combined_candidates["specific_note"].str.casefold().isin([
            "calabrian bergamot",
            "chinese black tea",
            "madagascar vanilla",
            "orange blossom",
            "virginia cedar",
            "white musk"
        ])
    ][
        ["specific_note", "possible_family", "sources",
         "matches_spacy_root", "review_status"]
    ]
)

,review_status,number_of_candidates
0,Wymaga weryfikacji,470
1,Priorytetowy kandydat,397


,specific_note,possible_family,sources,matches_spacy_root,review_status
146,calabrian bergamot,bergamot,spacy_root | word_combinations,True,Priorytetowy kandydat
183,chinese black tea,black,word_combinations,False,Wymaga weryfikacji
184,chinese black tea,black tea,word_combinations,False,Wymaga weryfikacji
185,chinese black tea,chinese tea,word_combinations,False,Wymaga weryfikacji
186,chinese black tea,tea,spacy_root | word_combinations,True,Priorytetowy kandydat
465,madagascar vanilla,vanilla,spacy_root | word_combinations,True,Priorytetowy kandydat
529,orange blossom,orange,word_combinations,False,Wymaga weryfikacji
766,virginia cedar,cedar,spacy_root | word_combinations,True,Priorytetowy kandydat
817,white musk,musk,spacy_root | word_combinations,True,Priorytetowy kandydat


In [49]:
candidate_counts = (
    combined_candidates
    .groupby("specific_note")
    .agg(
        number_of_candidates=("possible_family", "nunique"),
        has_spacy_candidate=("matches_spacy_root", "any")
    )
)

all_notes_index = pd.Index(notes, name="specific_note")

candidate_counts = candidate_counts.reindex(
    all_notes_index,
    fill_value=0
)

candidate_counts["number_of_candidates"] = (
    candidate_counts["number_of_candidates"].fillna(0).astype(int)
)

candidate_counts["has_spacy_candidate"] = (
    candidate_counts["has_spacy_candidate"].fillna(False).astype(bool)
)

candidate_counts["candidate_group"] = pd.cut(
    candidate_counts["number_of_candidates"],
    bins=[-1, 0, 1, 3, float("inf")],
    labels=[
        "Brak kandydatów",
        "Jeden kandydat",
        "2–3 kandydatów",
        "4 lub więcej"
    ]
)

display(
    candidate_counts["candidate_group"]
    .value_counts(sort=False)
    .rename_axis("candidate_group")
    .reset_index(name="number_of_notes")
)

print("Wszystkie unikalne nuty:", len(all_notes_index))
print("Nuty z przynajmniej jednym kandydatem:",
      (candidate_counts["number_of_candidates"] > 0).sum())
print("Nuty bez kandydatów:",
      (candidate_counts["number_of_candidates"] == 0).sum())

,candidate_group,number_of_notes
0,Brak kandydatów,935
1,Jeden kandydat,622
2,2–3 kandydatów,100
3,4 lub więcej,7


Wszystkie unikalne nuty: 1664
Nuty z przynajmniej jednym kandydatem: 729
Nuty bez kandydatów: 935


In [50]:
# 1. Nuty, dla których nie znaleziono żadnego kandydata
notes_without_candidates = candidate_counts[
    candidate_counts["number_of_candidates"] == 0
].index.tolist()

print("Przykładowe nuty bez kandydatów:")
display(
    pd.DataFrame({"note": sorted(notes_without_candidates)})
    .sample(
        n=min(60, len(notes_without_candidates)),
        random_state=42
    )
    .sort_values("note")
    .reset_index(drop=True)
)

# 2. Nuty, dla których znaleziono dokładnie jednego kandydata
notes_with_one_candidate = candidate_counts[
    candidate_counts["number_of_candidates"] == 1
].index

single_candidate_examples = (
    combined_candidates[
        combined_candidates["specific_note"].isin(notes_with_one_candidate)
    ][
        ["specific_note", "possible_family", "sources",
         "matches_spacy_root", "review_status"]
    ]
    .sample(
        n=min(30, len(combined_candidates[
            combined_candidates["specific_note"].isin(notes_with_one_candidate)
        ])),
        random_state=42
    )
    .sort_values("specific_note")
    .reset_index(drop=True)
)

print("Przykłady nut z jednym kandydatem:")
display(single_candidate_examples)

Przykładowe nuty bez kandydatów:


,note
0,ambergris
1,ambrettolide
2,angels trumpet
3,balsamic notes
4,baobab
5,barley
6,bigarade
7,blue margarita
8,buchu or agathosma
9,cade oil


Przykłady nut z jednym kandydatem:


,specific_note,possible_family,sources,matches_spacy_root,review_status
0,african ginger,ginger,spacy_root | word_combinations,True,Priorytetowy kandydat
1,black diamond,black,word_combinations,False,Wymaga weryfikacji
2,cherry liqueur,cherry,word_combinations,False,Wymaga weryfikacji
3,coconut leaf,coconut,word_combinations,False,Wymaga weryfikacji
4,coffee tincture,coffee,word_combinations,False,Wymaga weryfikacji
5,cookie dough,cookie,word_combinations,False,Wymaga weryfikacji
6,crystal amber,amber,spacy_root | word_combinations,True,Priorytetowy kandydat
7,ebony tree,ebony,word_combinations,False,Wymaga weryfikacji
8,eglantine rose,rose,word_combinations,False,Wymaga weryfikacji
9,freesia petals,freesia,word_combinations,False,Wymaga weryfikacji


In [51]:
output_dir = Path("../dane/slownik/")

priority_notes = set(
    combined_candidates.loc[
        combined_candidates["review_status"] == "Priorytetowy kandydat",
        "specific_note"
    ]
)

candidate_details = (
    combined_candidates
    .groupby("specific_note", sort=True)
    .apply(
        lambda group: pd.Series({
            "proposed_families": " | ".join(
                group["possible_family"].drop_duplicates().tolist()
            ),
            "candidate_details": "\n".join(
                f'{row["possible_family"]} '
                f'[źródła: {row["sources"]}; '
                f'status: {row["review_status"]}]'
                for _, row in group.iterrows()
            )
        }),
        include_groups=False
    )
    .reset_index()
)

all_notes_df = pd.DataFrame({
    "original_note": notes
}).drop_duplicates()

all_notes_df["number_of_candidates"] = (
    all_notes_df["original_note"]
    .map(candidate_counts["number_of_candidates"])
    .fillna(0)
    .astype(int)
)

all_notes_df["has_priority_candidate"] = (
    all_notes_df["original_note"].isin(priority_notes)
)

all_notes_df = all_notes_df.merge(
    candidate_details,
    left_on="original_note",
    right_on="specific_note",
    how="left"
).drop(columns=["specific_note"])

no_candidates_df = all_notes_df[
    all_notes_df["number_of_candidates"] == 0
].copy()

quick_review_df = all_notes_df[
    (all_notes_df["number_of_candidates"] > 0)
    & all_notes_df["has_priority_candidate"]
].copy()

detailed_review_df = all_notes_df[
    (all_notes_df["number_of_candidates"] > 0)
    & ~all_notes_df["has_priority_candidate"]
].copy()

print("Folder docelowy:", output_dir.resolve())
print("Bez kandydatów:", len(no_candidates_df))
print("Szybka weryfikacja:", len(quick_review_df))
print("Dokładna weryfikacja:", len(detailed_review_df))
print(
    "Suma:",
    len(no_candidates_df) + len(quick_review_df) + len(detailed_review_df)
)

Folder docelowy: C:\Users\Adam\Desktop\PerfumeCollection\dane\slownik
Bez kandydatów: 935
Szybka weryfikacja: 397
Dokładna weryfikacja: 332
Suma: 1664


In [52]:
no_candidates_export = no_candidates_df[
    ["original_note"]
].copy()

no_candidates_export["selected_family"] = (
    no_candidates_export["original_note"]
)

no_candidates_export["review_status"] = "Domyślnie bez zmian"

no_candidates_export = no_candidates_export.sort_values(
    "original_note"
).reset_index(drop=True)

no_candidates_export.to_csv(
    output_dir / "nuty_bez_kandydatow.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Zapisano {len(no_candidates_export)} nut do:",
    output_dir / "nuty_bez_kandydatow.csv"
)

Zapisano 935 nut do: ..\dane\slownik\nuty_bez_kandydatow.csv


In [53]:
quick_review_export = quick_review_df[
    [
        "original_note",
        "proposed_families",
        "candidate_details",
        "number_of_candidates"
    ]
].copy()

quick_review_export["selected_family"] = ""
quick_review_export["review_status"] = "Do szybkiej weryfikacji"

quick_review_export = quick_review_export.sort_values(
    "original_note"
).reset_index(drop=True)

quick_review_export.to_csv(
    output_dir / "nuty_szybka_weryfikacja.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Zapisano {len(quick_review_export)} nut do:",
    output_dir / "nuty_szybka_weryfikacja.csv"
)

Zapisano 397 nut do: ..\dane\slownik\nuty_szybka_weryfikacja.csv


In [54]:
detailed_review_export = detailed_review_df[
    [
        "original_note",
        "proposed_families",
        "candidate_details",
        "number_of_candidates"
    ]
].copy()

detailed_review_export["selected_family"] = ""
detailed_review_export["review_status"] = "Do dokładnej weryfikacji"

detailed_review_export = detailed_review_export.sort_values(
    "original_note"
).reset_index(drop=True)

detailed_review_export.to_csv(
    output_dir / "nuty_dokladna_weryfikacja.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Zapisano {len(detailed_review_export)} nut do:",
    output_dir / "nuty_dokladna_weryfikacja.csv"
)

Zapisano 332 nut do: ..\dane\slownik\nuty_dokladna_weryfikacja.csv
